# SWE-bench Style GRPO Training with Qwen2-0.5B-InstructTrain a small language model to generate correct code patches for simple bug fixes using GRPO.- **Model**: Qwen/Qwen2-0.5B-Instruct- **Task**: Code patch generation (off-by-one, missing returns, wrong variables, wrong conditionals)- **Method**: TRL GRPOTrainer with LoRA (rank 32)- **Logging**: Weights & Biases (project: tinker-rl-agentic)- **Hub**: arvindcr4/

In [ ]:
# Cell 1: Install dependencies!pip install -q trl transformers peft datasets accelerate bitsandbytes wandb huggingface_hub

In [ ]:
# Cell 2: GPU checkimport torchprint(f"CUDA available: {torch.cuda.is_available()}")print(f"GPU: {torch.cuda.get_device_name(0)}")print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Cell 3: Auto-login with pre-configured keys
import os

# Set your API keys here or use Colab secrets
# In Colab: Runtime > Manage secrets, add WANDB_API_KEY and HF_TOKEN
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except:
    # Fallback: set manually
    os.environ.setdefault('WANDB_API_KEY', 'YOUR_WANDB_API_KEY')
    os.environ.setdefault('HF_TOKEN', 'YOUR_HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

In [ ]:
# Cell 4: Define code fix examples (SWE-bench style)import json# 30 examples of (buggy code, fix description, correct patch)TRAINING_DATA = [    # Off-by-one errors    {"prompt": "Fix the off-by-one error in this function:\n```python\ndef get_elements(lst, n):\n    return lst[:n-1]\n```\nThe function should return the first n elements.", "patch": "def get_elements(lst, n):\n    return lst[:n]", "category": "off-by-one"},    {"prompt": "Fix the off-by-one error:\n```python\ndef sum_range(start, end):\n    total = 0\n    for i in range(start, end-1):\n        total += i\n    return total\n```\nShould sum from start to end inclusive.", "patch": "def sum_range(start, end):\n    total = 0\n    for i in range(start, end+1):\n        total += i\n    return total", "category": "off-by-one"},    {"prompt": "Fix the off-by-one error:\n```python\ndef last_n_elements(lst, n):\n    return lst[len(lst)-n+1:]\n```\nShould return the last n elements.", "patch": "def last_n_elements(lst, n):\n    return lst[len(lst)-n:]", "category": "off-by-one"},    {"prompt": "Fix the off-by-one in the binary search:\n```python\ndef binary_search(arr, target):\n    lo, hi = 0, len(arr)\n    while lo < hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            lo = mid\n        else:\n            hi = mid\n    return -1\n```\nlo should advance past mid.", "patch": "def binary_search(arr, target):\n    lo, hi = 0, len(arr)\n    while lo < hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            lo = mid + 1\n        else:\n            hi = mid\n    return -1", "category": "off-by-one"},    {"prompt": "Fix the fence post error:\n```python\ndef count_segments(n):\n    return n\n```\nn points create n-1 segments.", "patch": "def count_segments(n):\n    return n - 1", "category": "off-by-one"},    # Missing return statements    {"prompt": "Fix the missing return statement:\n```python\ndef add(a, b):\n    result = a + b\n```\nThe function should return the sum.", "patch": "def add(a, b):\n    result = a + b\n    return result", "category": "missing-return"},    {"prompt": "Fix the missing return:\n```python\ndef is_even(n):\n    if n % 2 == 0:\n        return True\n    else:\n        pass\n```\nShould return False for odd numbers.", "patch": "def is_even(n):\n    if n % 2 == 0:\n        return True\n    else:\n        return False", "category": "missing-return"},    {"prompt": "Fix the missing return:\n```python\ndef find_max(lst):\n    if not lst:\n        return None\n    max_val = lst[0]\n    for item in lst[1:]:\n        if item > max_val:\n            max_val = item\n```\nShould return the maximum value.", "patch": "def find_max(lst):\n    if not lst:\n        return None\n    max_val = lst[0]\n    for item in lst[1:]:\n        if item > max_val:\n            max_val = item\n    return max_val", "category": "missing-return"},    {"prompt": "Fix the missing return:\n```python\ndef factorial(n):\n    if n <= 1:\n        return 1\n    else:\n        factorial(n - 1) * n\n```\nRecursive call result is not returned.", "patch": "def factorial(n):\n    if n <= 1:\n        return 1\n    else:\n        return factorial(n - 1) * n", "category": "missing-return"},    {"prompt": "Fix the missing return:\n```python\ndef get_greeting(name):\n    greeting = f'Hello, {name}!'\n```\nShould return the greeting string.", "patch": "def get_greeting(name):\n    greeting = f'Hello, {name}!'\n    return greeting", "category": "missing-return"},    {"prompt": "Fix the missing return:\n```python\ndef reverse_string(s):\n    s[::-1]\n```\nShould return the reversed string.", "patch": "def reverse_string(s):\n    return s[::-1]", "category": "missing-return"},    # Wrong variable names    {"prompt": "Fix the wrong variable name:\n```python\ndef calculate_area(width, height):\n    return width * heigth\n```\nTypo in 'height'.", "patch": "def calculate_area(width, height):\n    return width * height", "category": "wrong-variable"},    {"prompt": "Fix the wrong variable name:\n```python\ndef greet(first_name, last_name):\n    return f'Hello, {first_name} {lst_name}!'\n```\nTypo in 'last_name'.", "patch": "def greet(first_name, last_name):\n    return f'Hello, {first_name} {last_name}!'", "category": "wrong-variable"},    {"prompt": "Fix the wrong variable name:\n```python\ndef process_data(input_data):\n    result = []\n    for item in input_data:\n        result.append(item * 2)\n    return results\n```\nShould be 'result' not 'results'.", "patch": "def process_data(input_data):\n    result = []\n    for item in input_data:\n        result.append(item * 2)\n    return result", "category": "wrong-variable"},    {"prompt": "Fix the wrong variable name:\n```python\ndef compute_average(numbers):\n    total = sum(numbers)\n    count = len(numbers)\n    return totl / count\n```\nTypo: 'totl' should be 'total'.", "patch": "def compute_average(numbers):\n    total = sum(numbers)\n    count = len(numbers)\n    return total / count", "category": "wrong-variable"},    {"prompt": "Fix the wrong variable name:\n```python\ndef merge_lists(list_a, list_b):\n    combined = list_a + list_c\n    return combined\n```\n'list_c' should be 'list_b'.", "patch": "def merge_lists(list_a, list_b):\n    combined = list_a + list_b\n    return combined", "category": "wrong-variable"},    {"prompt": "Fix the wrong variable name:\n```python\ndef format_name(first, last):\n    full_name = first + ' ' + lsat\n    return full_name\n```\n'lsat' should be 'last'.", "patch": "def format_name(first, last):\n    full_name = first + ' ' + last\n    return full_name", "category": "wrong-variable"},    # Incorrect conditionals    {"prompt": "Fix the incorrect conditional:\n```python\ndef is_positive(n):\n    if n < 0:\n        return True\n    return False\n```\nCondition is inverted.", "patch": "def is_positive(n):\n    if n > 0:\n        return True\n    return False", "category": "wrong-conditional"},    {"prompt": "Fix the incorrect conditional:\n```python\ndef can_vote(age):\n    if age < 18:\n        return True\n    return False\n```\nShould check age >= 18.", "patch": "def can_vote(age):\n    if age >= 18:\n        return True\n    return False", "category": "wrong-conditional"},    {"prompt": "Fix the incorrect conditional:\n```python\ndef is_empty(lst):\n    if len(lst) > 0:\n        return True\n    return False\n```\nLogic is inverted.", "patch": "def is_empty(lst):\n    if len(lst) == 0:\n        return True\n    return False", "category": "wrong-conditional"},    {"prompt": "Fix the incorrect conditional:\n```python\ndef is_valid_age(age):\n    if age < 0 and age > 150:\n        return False\n    return True\n```\nShould use 'or' not 'and'.", "patch": "def is_valid_age(age):\n    if age < 0 or age > 150:\n        return False\n    return True", "category": "wrong-conditional"},    {"prompt": "Fix the incorrect conditional:\n```python\ndef check_bounds(x, lower, upper):\n    if x >= lower or x <= upper:\n        return True\n    return False\n```\nShould use 'and' not 'or'.", "patch": "def check_bounds(x, lower, upper):\n    if x >= lower and x <= upper:\n        return True\n    return False", "category": "wrong-conditional"},    {"prompt": "Fix the incorrect conditional:\n```python\ndef needs_update(version, min_version):\n    if version >= min_version:\n        return True\n    return False\n```\nNeeds update when version is LESS than min.", "patch": "def needs_update(version, min_version):\n    if version < min_version:\n        return True\n    return False", "category": "wrong-conditional"},    # Mixed bug types    {"prompt": "Fix the bug:\n```python\ndef safe_divide(a, b):\n    if b == 0:\n        return a / b\n    return 0\n```\nThe branches are swapped.", "patch": "def safe_divide(a, b):\n    if b == 0:\n        return 0\n    return a / b", "category": "wrong-conditional"},    {"prompt": "Fix the bug:\n```python\ndef flatten(nested_list):\n    result = []\n    for sublist in nested_list:\n        for item in sublist:\n            result.append(item)\n```\nMissing return.", "patch": "def flatten(nested_list):\n    result = []\n    for sublist in nested_list:\n        for item in sublist:\n            result.append(item)\n    return result", "category": "missing-return"},    {"prompt": "Fix the bug:\n```python\ndef clamp(value, min_val, max_val):\n    if value < min_val:\n        return min_val\n    if value > max_val:\n        return min_val\n    return value\n```\nSecond return should be max_val.", "patch": "def clamp(value, min_val, max_val):\n    if value < min_val:\n        return min_val\n    if value > max_val:\n        return max_val\n    return value", "category": "wrong-variable"},    {"prompt": "Fix the bug:\n```python\ndef remove_duplicates(lst):\n    seen = set()\n    result = []\n    for item in lst:\n        if item not in seen:\n            seen.add(item)\n    return result\n```\nForgot to append item to result.", "patch": "def remove_duplicates(lst):\n    seen = set()\n    result = []\n    for item in lst:\n        if item not in seen:\n            seen.add(item)\n            result.append(item)\n    return result", "category": "missing-logic"},    {"prompt": "Fix the bug:\n```python\ndef celsius_to_fahrenheit(c):\n    return c * 9/5 - 32\n```\nFormula should add 32, not subtract.", "patch": "def celsius_to_fahrenheit(c):\n    return c * 9/5 + 32", "category": "wrong-operator"},    {"prompt": "Fix the bug:\n```python\ndef power(base, exp):\n    if exp == 0:\n        return 0\n    return base * power(base, exp - 1)\n```\nBase case should return 1, not 0.", "patch": "def power(base, exp):\n    if exp == 0:\n        return 1\n    return base * power(base, exp - 1)", "category": "wrong-value"},    {"prompt": "Fix the bug:\n```python\ndef count_vowels(s):\n    count = 0\n    vowels = 'aeiou'\n    for char in s:\n        if char in vowels:\n            count += 1\n    return s\n```\nShould return count, not s.", "patch": "def count_vowels(s):\n    count = 0\n    vowels = 'aeiou'\n    for char in s:\n        if char in vowels:\n            count += 1\n    return count", "category": "wrong-variable"}]print(f"Created {len(TRAINING_DATA)} SWE-bench style training examples")categories = {}for ex in TRAINING_DATA:    cat = ex["category"]    categories[cat] = categories.get(cat, 0) + 1print(f"\nCategories: {dict(categories)}")

In [ ]:
# Cell 5: Define reward function for code patchesimport jsonimport reimport difflibdef extract_patch(text):    """Extract code patch from model output."""    # Look for ```python ... ``` blocks    match = re.search(r'```python\s*\n(.*?)\n```', text, re.DOTALL)    if match:        return match.group(1).strip()    # Look for ``` ... ``` blocks    match = re.search(r'```\s*\n(.*?)\n```', text, re.DOTALL)    if match:        return match.group(1).strip()    # Look for def ... blocks    match = re.search(r'(def\s+\w+.*?)(?:\n\n|\Z)', text, re.DOTALL)    if match:        return match.group(1).strip()    return Nonedef normalize_code(code):    """Normalize code for comparison."""    if code is None:        return ""    # Remove extra whitespace, normalize indentation    lines = code.strip().split('\n')    lines = [line.rstrip() for line in lines]    return '\n'.join(lines)def swebench_reward_fn(completions, **kwargs):    """    Reward function for SWE-bench style GRPO.    - 1.0: patch matches expected fix (normalized comparison)    - 0.5: patch is valid Python and contains key fix elements    - 0.0: invalid or no patch    """    prompts = kwargs.get("prompts", [])    rewards = []    for i, completion in enumerate(completions):        if isinstance(completion, list):            text = completion[-1]["content"] if completion else ""        else:            text = completion                patch = extract_patch(text)        if patch is None:            rewards.append(0.0)            continue                # Find expected patch        prompt_text = prompts[i] if i < len(prompts) else ""        if isinstance(prompt_text, list):            prompt_text = prompt_text[-1]["content"] if prompt_text else ""                expected_patch = None        for example in TRAINING_DATA:            if example["prompt"][:50] in prompt_text:                expected_patch = example["patch"]                break                if expected_patch is None:            rewards.append(0.0)            continue                norm_patch = normalize_code(patch)        norm_expected = normalize_code(expected_patch)                # Exact match (after normalization)        if norm_patch == norm_expected:            rewards.append(1.0)        else:            # Similarity-based partial credit            ratio = difflib.SequenceMatcher(None, norm_patch, norm_expected).ratio()            if ratio > 0.8:                rewards.append(0.75)            elif ratio > 0.5:                rewards.append(0.5)            elif "def " in patch:                rewards.append(0.1)            else:                rewards.append(0.0)        return rewardsprint("SWE-bench reward function defined.")print("  1.0  = exact patch match")print("  0.75 = very similar patch (>80% similarity)")print("  0.5  = moderately similar patch (>50% similarity)")print("  0.1  = valid Python function but wrong fix")print("  0.0  = invalid output")

In [ ]:
# Cell 6: Format dataset for GRPOTrainerfrom datasets import Datasetdef format_prompt(example):    """Format a training example into a chat-style prompt."""    system_msg = (        "You are an expert software engineer. "        "When given buggy code and a description of the bug, provide the corrected code.\n"        "Format your response as:\n"        "```python\n<corrected code>\n```\n"        "Only output the corrected function, nothing else."    )    return [        {"role": "system", "content": system_msg},        {"role": "user", "content": example["prompt"]}    ]formatted_data = []for example in TRAINING_DATA:    formatted_data.append({        "prompt": format_prompt(example),        "expected_patch": example["patch"],        "category": example["category"]    })dataset = Dataset.from_list(formatted_data)print(f"Dataset size: {len(dataset)}")print(f"Sample prompt:\n{dataset[0]['prompt'][-1]['content'][:200]}...")

In [ ]:
# Cell 7: Load model and tokenizerfrom transformers import AutoModelForCausalLM, AutoTokenizerMODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenmodel = AutoModelForCausalLM.from_pretrained(    MODEL_NAME,    torch_dtype=torch.bfloat16,    device_map="auto",    trust_remote_code=True)print(f"Model loaded: {MODEL_NAME}")print(f"Parameters: {model.num_parameters():,}")

In [ ]:
from peft import LoraConfig# Cell 8: LoRA configurationlora_config = LoraConfig(    r=32,    lora_alpha=64,    lora_dropout=0.05,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],    bias="none",    task_type="CAUSAL_LM")print(f"LoRA config: rank={lora_config.r}, alpha={lora_config.lora_alpha}")print(f"Target modules: {lora_config.target_modules}")

In [ ]:
# Cell 9: GRPO configurationfrom trl import GRPOConfiggrpo_config = GRPOConfig(    output_dir="./grpo-swebench-qwen0.5b",    num_train_epochs=5,    per_device_train_batch_size=4,    gradient_accumulation_steps=2,    learning_rate=4e-5,    lr_scheduler_type="cosine",    warmup_ratio=0.1,    max_completion_length=256,    num_generations=16,    logging_steps=1,    save_steps=50,    bf16=True,    report_to="wandb",    run_name="swebench-grpo-qwen0.5b",    seed=42,    remove_unused_columns=False)print(f"GRPO Config:")print(f"  Epochs: {grpo_config.num_train_epochs}")print(f"  Batch size: {grpo_config.per_device_train_batch_size}")print(f"  LR: {grpo_config.learning_rate}")print(f"  Group size: {grpo_config.num_generations}")

In [ ]:
# Cell 10: Create GRPOTrainer and trainfrom trl import GRPOTraineros.environ["WANDB_PROJECT"] = "tinker-rl-agentic"trainer = GRPOTrainer(    model=model,    args=grpo_config,    train_dataset=dataset,    reward_funcs=swebench_reward_fn,    peft_config=lora_config,    processing_class=tokenizer)print("Starting GRPO training...")trainer.train()print("Training complete!")

In [ ]:
# Cell 11: Save the trained modelOUTPUT_DIR = "./grpo-swebench-qwen0.5b-final"trainer.save_model(OUTPUT_DIR)tokenizer.save_pretrained(OUTPUT_DIR)print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Cell 12: Evaluation - compare base vs GRPO-trained modelfrom transformers import pipelinetrained_pipe = pipeline(    "text-generation",    model=OUTPUT_DIR,    tokenizer=tokenizer,    device_map="auto",    torch_dtype=torch.bfloat16)base_pipe = pipeline(    "text-generation",    model=MODEL_NAME,    tokenizer=tokenizer,    device_map="auto",    torch_dtype=torch.bfloat16)# Test examples (not in training set)test_examples = [    {        "prompt": "Fix the bug:\n```python\ndef double_list(lst):\n    return [x * 3 for x in lst]\n```\nShould multiply by 2, not 3.",        "patch": "def double_list(lst):\n    return [x * 2 for x in lst]"    },    {        "prompt": "Fix the missing return:\n```python\ndef square(x):\n    x ** 2\n```\nShould return x squared.",        "patch": "def square(x):\n    return x ** 2"    },    {        "prompt": "Fix the wrong variable:\n```python\ndef concat(a, b):\n    return a + c\n```\n'c' should be 'b'.",        "patch": "def concat(a, b):\n    return a + b"    },    {        "prompt": "Fix the conditional:\n```python\ndef abs_value(x):\n    if x > 0:\n        return -x\n    return x\n```\nShould negate when x < 0.",        "patch": "def abs_value(x):\n    if x < 0:\n        return -x\n    return x"    },    {        "prompt": "Fix the off-by-one:\n```python\ndef nth_element(lst, n):\n    return lst[n]\n```\nShould be 1-indexed (first element is n=1).",        "patch": "def nth_element(lst, n):\n    return lst[n-1]"    }]print("=" * 80)print("EVALUATION: Base vs GRPO-trained Model (SWE-bench)")print("=" * 80)base_scores = []trained_scores = []for example in test_examples:    messages = format_prompt(example)    base_out = base_pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)    trained_out = trained_pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)        base_text = base_out[0]["generated_text"][-1]["content"] if isinstance(base_out[0]["generated_text"], list) else base_out[0]["generated_text"]    trained_text = trained_out[0]["generated_text"][-1]["content"] if isinstance(trained_out[0]["generated_text"], list) else trained_out[0]["generated_text"]        base_patch = extract_patch(base_text)    trained_patch = extract_patch(trained_text)        expected = normalize_code(example["patch"])        base_match = normalize_code(base_patch) == expected if base_patch else False    trained_match = normalize_code(trained_patch) == expected if trained_patch else False        base_scores.append(1 if base_match else 0)    trained_scores.append(1 if trained_match else 0)        print(f"\nPrompt: {example['prompt'][:60]}...")    print(f"  Base match:    {'YES' if base_match else 'NO'}")    print(f"  Trained match: {'YES' if trained_match else 'NO'}")print(f"\n{'=' * 80}")print(f"Base exact match:    {sum(base_scores)}/{len(test_examples)} ({100*sum(base_scores)/len(test_examples):.1f}%)")print(f"Trained exact match: {sum(trained_scores)}/{len(test_examples)} ({100*sum(trained_scores)/len(test_examples):.1f}%)")

In [ ]:
# Cell 13: Push to HuggingFace HubHUB_REPO = "arvindcr4/swebench-grpo-qwen0.5b"trainer.push_to_hub(HUB_REPO)tokenizer.push_to_hub(HUB_REPO)print(f"Model pushed to: https://huggingface.co/{HUB_REPO}")wandb.finish()print("Done! W&B run finished.")